In [1]:
from pyspark.sql import functions as F

# ============================================================================
# CONFIGURATION
# ============================================================================

CATALOG = "gitrepo"
SCHEMA = "default"

# Source Parquet paths
SILVER_AUDIT_PARQUET = "/Volumes/gitrepo/default/git_oci_aidp_silver/audit_logs/data"
SILVER_FLOW_PARQUET = "/Volumes/gitrepo/default/git_oci_aidp_silver/flow_logs/data"

# Target Delta tables
SILVER_AUDIT_TABLE = f"{CATALOG}.{SCHEMA}.silver_audit_logs"
SILVER_FLOW_TABLE = f"{CATALOG}.{SCHEMA}.silver_flow_logs"

# Spark tuning for large data
spark.conf.set("spark.sql.shuffle.partitions", "200")
spark.conf.set("spark.sql.adaptive.enabled", "true")
spark.conf.set("spark.sql.adaptive.coalescePartitions.enabled", "true")

print("=" * 70)
print("SILVER PARQUET TO DELTA CONVERSION")
print("=" * 70)

SILVER PARQUET TO DELTA CONVERSION


In [2]:
print("\n[1/4] Reading Silver Parquet...")
silver_audit_df = spark.read.parquet(SILVER_AUDIT_PARQUET)
record_count = silver_audit_df.count()
print(f"✓ Loaded {record_count:,} records")

print("\n[2/4] Writing to Delta table...")
(
    silver_audit_df.write
    .format("delta")
    .mode("overwrite")
    .partitionBy("ingest_date")
    .option("overwriteSchema", "true")
    .saveAsTable(SILVER_AUDIT_TABLE)
)
print(f"✓ Written to {SILVER_AUDIT_TABLE}")

print("\n[3/4] Optimizing with Z-ORDER...")
spark.sql(f"""
    OPTIMIZE {SILVER_AUDIT_TABLE}
    ZORDER BY (event_name, principal_id, ip_address)
""")
print("✓ Z-ORDER optimization complete")

print("\n[4/4] Computing statistics...")
spark.sql(f"ANALYZE TABLE {SILVER_AUDIT_TABLE} COMPUTE STATISTICS FOR ALL COLUMNS")
print("✓ Statistics computed")

print(f"\n✓ AUDIT LOGS CONVERSION COMPLETE: {record_count:,} records")


[1/4] Reading Silver Parquet...


✓ Loaded 17,383,006 records

[2/4] Writing to Delta table...


✓ Written to gitrepo.default.silver_audit_logs

[3/4] Optimizing with Z-ORDER...


✓ Z-ORDER optimization complete

[4/4] Computing statistics...


opc-request-id: csid86dedaa64272a72fb9712f3ac7d0/33c4ebcf90e64fbb8ddb1743eed3c73f/C14923358EBA452283AC73D9B7ED67C8

Command ID failed with java.lang.RuntimeException: java.lang.Exception: [---------------------------------------------------------------------------, Py4JJavaError                             Traceback (most recent call last), Cell In[17], line 25
     22 print("✓ Z-ORDER optimization complete")
     24 print("\n[4/4] Computing statistics...")
---> 25 spark.sql(f"ANALYZE TABLE {SILVER_AUDIT_TABLE} COMPUTE STATISTICS FOR ALL COLUMNS")
     26 print("✓ Statistics computed")
     28 print(f"\n✓ AUDIT LOGS CONVERSION COMPLETE: {record_count:,} records")
, File /opt/spark/python/lib/pyspark.zip/pyspark/sql/session.py:1631, in SparkSession.sql(self, sqlQuery, args, **kwargs)
   1627         assert self._jvm is not None
   1628         litArgs = self._jvm.PythonUtils.toArray(
   1629             [_to_java_column(lit(v)) for v in (args or [])]
   1630         )
-> 1631     return

In [3]:
print("\n[1/4] Reading Silver Parquet...")
silver_flow_df = spark.read.parquet(SILVER_FLOW_PARQUET)
flow_count = silver_flow_df.count()
print(f"✓ Loaded {flow_count:,} records")

print("\n[2/4] Writing to Delta table...")
(
    silver_flow_df.write
    .format("delta")
    .mode("overwrite")
    .partitionBy("ingest_date")
    .option("overwriteSchema", "true")
    .saveAsTable(SILVER_FLOW_TABLE)
)
print(f"✓ Written to {SILVER_FLOW_TABLE}")

print("\n[3/4] Optimizing with Z-ORDER...")
spark.sql(f"""
    OPTIMIZE {SILVER_FLOW_TABLE}
    ZORDER BY (src_ip, dst_ip, dst_port, action)
""")
print("✓ Z-ORDER optimization complete")

print("\n[4/4] Computing statistics...")
spark.sql(f"ANALYZE TABLE {SILVER_FLOW_TABLE} COMPUTE STATISTICS FOR ALL COLUMNS")
print("✓ Statistics computed")

print(f"\n✓ FLOW LOGS CONVERSION COMPLETE: {flow_count:,} records")


[1/4] Reading Silver Parquet...


✓ Loaded 161,667,060 records

[2/4] Writing to Delta table...


✓ Written to gitrepo.default.silver_flow_logs

[3/4] Optimizing with Z-ORDER...


✓ Z-ORDER optimization complete

[4/4] Computing statistics...


opc-request-id: csid86dedaa64272a72fb9712f3ac7d0/9e119c85165e4237a0ee4d8807503b49/3EC2E04CB28B4C02B27F80643147ADAF

Command ID failed with java.lang.RuntimeException: java.lang.Exception: [---------------------------------------------------------------------------, Py4JJavaError                             Traceback (most recent call last), Cell In[21], line 25
     22 print("✓ Z-ORDER optimization complete")
     24 print("\n[4/4] Computing statistics...")
---> 25 spark.sql(f"ANALYZE TABLE {SILVER_FLOW_TABLE} COMPUTE STATISTICS FOR ALL COLUMNS")
     26 print("✓ Statistics computed")
     28 print(f"\n✓ FLOW LOGS CONVERSION COMPLETE: {flow_count:,} records")
, File /opt/spark/python/lib/pyspark.zip/pyspark/sql/session.py:1631, in SparkSession.sql(self, sqlQuery, args, **kwargs)
   1627         assert self._jvm is not None
   1628         litArgs = self._jvm.PythonUtils.toArray(
   1629             [_to_java_column(lit(v)) for v in (args or [])]
   1630         )
-> 1631     return Dat